In [1]:
import xarray as xr
import numpy as np
import netCDF4 as nc

import sys
sys.path.insert(0,"../modules")
import fortsa

In [2]:
# Read detrended time series
dt_est = 15/1440

ds = xr.open_dataset("data/detrended_driver_timeseries.nc")
flow = ds['flow'].values
airt = ds['airt'].values
gs_salt = ds['gs_salt'].values
gs_temp = ds['gs_temp'].values
gs_tran = ds['gs_tran'].values
wind_u = ds['wind_u'].values
wind_v = ds['wind_v'].values
ds.close()

ds = xr.open_dataset("data/detrended_estuary_timeseries.nc")
zeta = ds['zeta'].values
salt = ds['salt'].values
temp = ds['temp'].values
ds.close()

In [3]:
flow.shape, airt.shape, gs_salt.shape, gs_temp.shape, gs_tran.shape, wind_u.shape, wind_v.shape

((385535,), (385535,), (385535,), (385535,), (385535,), (385535,), (385535,))

In [4]:
zeta.shape, salt.shape, temp.shape

((385535, 2), (385535, 3, 2), (385535, 3, 2))

In [ ]:
Nlag = int(365/dt_est)
cor_tran = np.empty([Nlag*2+1,2])
cor_flow = np.empty([Nlag*2+1,2])
cor_windu = np.empty([Nlag*2+1,2])
cor_windv = np.empty([Nlag*2+1,2])
for i in range(2):
    cor_tran[:,i] = fortsa.correlation(gs_tran, zeta[:,i], Nlag)
    cor_flow[:,i] = fortsa.correlation(flow, zeta[:,i], Nlag)
    cor_windu[:,i] = fortsa.correlation(wind_u[:-1], zeta[:,i], Nlag)
    cor_windv[:,i] = fortsa.correlation(wind_v[:-1], zeta[:,i], Nlag)

cor_gs_salt = np.empty([Nlag*2+1,3,2])
cor_flow_salt = np.empty([Nlag*2+1,3,2])
for i in range(3):
    for j in range(2):
        cor_gs_salt[:,i,j] = fortsa.correlation(gs_salt, salt[:,i,j], Nlag)
        cor_flow_salt[:,i,j] = fortsa.correlation(flow, salt[:,i,j], Nlag)

cor_gs_temp = np.empty([Nlag*2+1,3,2])
cor_flow_temp = np.empty([Nlag*2+1,3,2])
cor_airt = np.empty([Nlag*2+1,3,2])
for i in range(3):
    for j in range(2):
        cor_gs_temp[:,i,j] = fortsa.correlation(gs_temp, temp[:,i,j], Nlag)
        cor_flow_temp[:,i,j] = fortsa.correlation(flow, temp[:,i,j], Nlag)
        cor_airt[:,i,j] = fortsa.correlation(airt, temp[:,i,j], Nlag)

In [ ]:
lags = np.arange(-Nlag, Nlag+1) * dt_est

ds_out = nc.Dataset("data/lagged_xcorrelations.nc", mode='w', format='NETCDF4')
ds_out.createDimension("lags", lags.size)
ds_out.createDimension("two", 2)
ds_out.createDimension("three", 3)

ds_lags = ds_out.createVariable("lags", "f8", ("lags",))
ds_tran_zeta = ds_out.createVariable("cor_tran_zeta", "f8", ("lags","two",))
ds_flow_zeta = ds_out.createVariable("cor_flow_zeta", "f8", ("lags","two",))
ds_windu_zeta = ds_out.createVariable("cor_windu_zeta", "f8", ("lags","two",))
ds_windv_zeta = ds_out.createVariable("cor_windv_zeta", "f8", ("lags","two",))
ds_salt_salt = ds_out.createVariable("cor_salt_salt", "f8", ("lags","three","two",))
ds_flow_salt = ds_out.createVariable("cor_flow_salt", "f8", ("lags","three","two",))
ds_temp_temp = ds_out.createVariable("cor_temp_temp", "f8", ("lags","three","two",))
ds_flow_temp = ds_out.createVariable("cor_flow_temp", "f8", ("lags","three","two",))
ds_airt_temp = ds_out.createVariable("cor_airt_temp", "f8", ("lags","three","two",))

ds_lags[:] = lags
ds_tran_zeta[:] = cor_tran
ds_flow_zeta[:] = cor_flow
ds_salt_salt[:] = cor_gs_salt
ds_flow_salt[:] = cor_flow_salt
ds_temp_temp[:] = cor_gs_temp
ds_flow_temp[:] = cor_flow_temp
ds_airt_temp[:] = cor_airt

ds_out.close()

In [ ]:
lags = np.round(np.arange(-Nlag, Nlag+1) * dt_est,2)

print("Water Surface Elevations")
print("Steele Point (transport, fluvial inflow, wind u, wind v):")
i = 0
indx1 = np.where(np.abs(cor_tran[:,i]) == np.abs(cor_tran[:,i]).max())[0][0]
indx2 = np.where(np.abs(cor_flow[:,i]) == np.abs(cor_flow[:,i]).max())[0][0]
indx3 = np.where(np.abs(cor_windu[:,i]) == np.abs(cor_windu[:,i]).max())[0][0]
indx4 = np.where(np.abs(cor_windv[:,i]) == np.abs(cor_windv[:,i]).max())[0][0]
print(f"{np.round(cor_tran[indx1,i],3)}, {lags[indx1]} | {np.round(cor_flow[indx2,i],3)}, {lags[indx2]} | {np.round(cor_windu[indx3,i],3)}, {lags[indx3]} | {np.round(cor_windv[indx4,i],3)}, {lags[indx4]}")
print('--')
print(f"{np.round(cor_tran[indx1,i],3)}, {lags[indx1]}")

print("Speedy Point:")
i = 1
indx1 = np.where(np.abs(cor_tran[:,i]) == np.abs(cor_tran[:,i]).max())[0][0]
indx2 = np.where(np.abs(cor_flow[:,i]) == np.abs(cor_flow[:,i]).max())[0][0]
indx3 = np.where(np.abs(cor_windu[:,i]) == np.abs(cor_windu[:,i]).max())[0][0]
indx4 = np.where(np.abs(cor_windv[:,i]) == np.abs(cor_windv[:,i]).max())[0][0]
print(f"{np.round(cor_tran[indx1,i],3)}, {lags[indx1]} | {np.round(cor_flow[indx2,i],3)}, {lags[indx2]} | {np.round(cor_windu[indx3,i],3)}, {lags[indx3]} | {np.round(cor_windv[indx4,i],3)}, {lags[indx4]}")
print('--')
print(f"{np.round(cor_tran[indx1,i],3)}, {lags[indx1]} ")

print("========================================================================================")
print("Salinity")
print("Steele Point Top (GS Salt, Fluvial Inflow):")
i, j = 0,0
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")
print("Steele Point Bot:")
i, j = 0,1
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")
print("----------------")
print("Speedy Point Top:")
i, j = 1,0
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")
print("Speedy Point Bot:")
i, j = 1,1
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")
print("----------------")
print("HR1 Top:")
i, j = 2,0
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")
print("HR1 Bot:")
i, j = 2,1
indx1 = np.where(np.abs(cor_gs_salt[:,i,j]) == np.abs(cor_gs_salt[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_salt[:,i,j]) == np.abs(cor_flow_salt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_salt[indx2,i,j],3)}, {lags[indx2]}")
print('--')
print(f"{np.round(cor_gs_salt[indx1,i,j],3)}, {lags[indx1]}")

print("========================================================================================")
print("Temperature")
print("Steele Point Top (GS Temp, Fluvial Inflow, Air Temp):")
i, j = 0,0
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")
print("Steele Point Bot:")
i, j = 0,1
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")
print("----------------")
print("Speedy Point Top:")
i, j = 1,0
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")
print("Speedy Point Bot:")
i, j = 1,1
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")
print("----------------")
print("HR1 Top:")
i, j = 2,0
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")
print("HR1 Bot:")
i, j = 2,1
indx1 = np.where(np.abs(cor_gs_temp[:,i,j]) == np.abs(cor_gs_temp[:,i,j]).max())[0][0]
indx2 = np.where(np.abs(cor_flow_temp[:,i,j]) == np.abs(cor_flow_temp[:,i,j]).max())[0][0]
indx3 = np.where(np.abs(cor_airt[:,i,j]) == np.abs(cor_airt[:,i,j]).max())[0][0]
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]} | {np.round(cor_flow_temp[indx2,i,j],3)}, {lags[indx2]} | {np.round(cor_airt[indx3,i,j],3)}, {lags[indx3]}")
print('--')
print(f"{np.round(cor_gs_temp[indx1,i,j],3)}, {lags[indx1]}")